In [ ]:
# 🌌 HoroConsultant - Production Cloud Fine-Tuning Pipeline
import os
import sys
import subprocess

# Suppress PyDev / frozen modules debugger warnings
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYTHONWARNINGS'] = 'ignore'

# 1. Load Secrets safely from Kaggle Secrets (individual try-except per key)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_key in ['HF_TOKEN', 'APP_SUPABASE_URL', 'APP_SUPABASE_KEY', 'GH_TOKEN']:
        try:
            val = user_secrets.get_secret(secret_key)
            if val:
                os.environ[secret_key] = val
                print(f'✅ Kaggle Secret loaded: {secret_key}')
        except Exception as e:
            print(f'ℹ️ Kaggle Secret note ({secret_key}): {e}')
except Exception as e:
    print(f'ℹ️ Kaggle Secrets Client not available: {e}')

# 2. Safe Git Clone / Pull with pure Python subprocess
target_dir = '/kaggle/working/HoroConsultant'
if not os.path.exists(target_dir):
    print('📦 Cloning HoroConsultant repository...')
    subprocess.run(['git', 'clone', 'https://github.com/pphothidaen/HoroConsultant.git', target_dir], check=True)
else:
    print('🔄 Resetting and pulling latest updates...')
    subprocess.run(['git', '-C', target_dir, 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', target_dir, 'reset', '--hard', 'origin/main'], check=True)

os.chdir(target_dir)
if target_dir not in sys.path:
    sys.path.insert(0, target_dir)

# 3. Install Fine-Tuning Dependencies preserving Kaggle's pre-installed CUDA PyTorch
print('📦 Checking pre-installed PyTorch & CUDA status...')
import torch
print(f'⚡ Kaggle PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   Device 0: {torch.cuda.get_device_name(0)}')
print('📦 Installing fine-tuning packages...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary', 'transformers>=4.40.0', 'peft>=0.10.0', 'bitsandbytes>=0.43.3', 'datasets>=2.18.0', 'trl>=0.12.0', 'huggingface_hub', 'accelerate'], check=True)
import torch
print(f'✅ Verified post-install PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

# 4. Run Cloud Training Orchestrator with execution logging
print('🚀 Launching Cloud Training Orchestrator...')
log_path = '/kaggle/working/train_execution.log'
proc = subprocess.Popen([sys.executable, 'scripts/cloud_train_orchestrator.py', '--platform', 'KAGGLE_T4', '--base-model', 'Qwen/Qwen2.5-7B-Instruct', '--epochs', '3'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(log_path, 'w', encoding='utf-8') as log_f:
    for line in iter(proc.stdout.readline, ''):
        sys.stdout.write(line)
        log_f.write(line)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'❌ Training orchestrator failed with exit code {proc.returncode}')
print('🎉 Training pipeline completed successfully!')
